# [센서 실습 1] 온도 센서 – DS18B20

디지털 온도 센서 **DS18B20**을 라즈베리파이의 1-Wire 인터페이스로 읽어 하루 동안의 실내 온도를 기록하고, 과열 구간을 찾아냅니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
DS18B20은 칩 안의 온도 감지 회로가 온도를 측정해 **디지털 값으로 바꿔서** 보내 주는 센서입니다.
- 선 하나(DATA)로 통신하는 **1-Wire** 방식 → 한 핀에 여러 개의 센서를 연결할 수 있음 (센서마다 고유 64비트 주소)
- 라즈베리파이 커널 드라이버가 센서를 읽어 `/sys/bus/w1/devices/28-xxxx/w1_slave` 라는 **텍스트 파일**로 보여 줌
- 파일의 첫 줄 끝 `YES`는 CRC(오류 검사) 통과, 둘째 줄의 `t=23125`는 23.125°C를 뜻함

### 부품 사양
| 항목 | 내용 |
|---|---|
| 측정 범위 | -55 ~ +125°C |
| 정확도 | ±0.5°C (-10 ~ +85°C) |
| 분해능 | 9~12비트 (12비트: 0.0625°C) |
| 변환 시간 | 최대 750ms (12비트) |
| 전원 | 3.0 ~ 5.5V |
| 출력 | 디지털 (1-Wire) |

## 2. 라즈베리파이 연결 방법
**준비물**: DS18B20 (방수형 프로브 권장), 4.7kΩ 저항 1개, 브레드보드, 점퍼선

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| DS18B20 VDD (빨강) | 3.3V (1번 핀) | 전원 |
| DS18B20 GND (검정) | GND (6번 핀) |  |
| DS18B20 DATA (노랑) | GPIO4 (7번 핀) | 1-Wire 데이터 |
| 4.7kΩ 저항 | DATA ↔ 3.3V 사이 | 풀업 저항 (필수) |



**인터페이스 활성화**
- **1-Wire**: `sudo raspi-config` → Interface Options → **1-Wire** → Enable 후 재부팅

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
# 추가 설치 없음 – 커널 드라이버가 파일로 제공
ls /sys/bus/w1/devices/      # 28-로 시작하는 폴더가 보이면 성공
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
센서가 내보내는 `w1_slave` 텍스트를 해석하여 섭씨 온도로 바꿉니다. CRC 검사가 실패하면 `None`을 돌려줍니다.

In [ ]:
def parse_w1_slave(text):
    """DS18B20 w1_slave 파일 내용 → 섭씨 온도 (CRC 오류면 None)"""
    lines = text.strip().split("\n")
    if not lines[0].strip().endswith("YES"):      # 첫 줄 끝이 YES 가 아니면 통신 오류
        return None
    pos = lines[1].find("t=")
    return int(lines[1][pos + 2:]) / 1000.0       # t=23125 → 23.125°C

print(parse_w1_slave("72 01 4b 46 7f ff 0e 10 57 : crc=57 YES\n72 01 4b 46 7f ff 0e 10 57 t=23125"))

### 4-2. 실제 센서 클래스
센서 폴더를 찾아 `w1_slave` 파일을 읽고, 위의 변환식으로 온도를 구합니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealDS18B20:
    """라즈베리파이 1-Wire 로 연결된 DS18B20 읽기"""
    def __init__(self):
        import glob
        devices = glob.glob("/sys/bus/w1/devices/28-*")
        if not devices:
            raise RuntimeError("DS18B20을 찾을 수 없습니다. 배선, 4.7kΩ 풀업 저항, 1-Wire 설정을 확인하세요.")
        self.path = devices[0] + "/w1_slave"

    def read(self):
        with open(self.path) as f:
            text = f.read()
        return {"temp_c": parse_w1_slave(text)}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 항목 | 가정한 값 | 근거 |
|---|---|---|
| 하루 온도 변화 | 20°C ± 7°C, 오후 3시 최고 · 새벽 3시 최저 | 냉난방이 없는 실내의 일반적인 일교차 |
| 일시 과열 | 13~15시에 +3°C | 창가 직사광선을 받는 상황 가정 |
| 측정 잡음 | 표준편차 0.2°C | 데이터시트 정확도 ±0.5°C 이내 |
| 분해능 | 0.0625°C 단위로 반올림 | 12비트 모드의 실제 분해능 |
| 통신 오류 | 1% 확률로 CRC 실패(`NO`) | 긴 배선에서 가끔 발생하는 현상 재현 |

가상 센서는 온도 숫자를 바로 돌려주지 않고, **실제 센서와 똑같은 `w1_slave` 텍스트**를 만들어 변환식에 넣습니다.

In [ ]:
class VirtualDS18B20:
    """DS18B20 가상 센서 – 실제와 같은 w1_slave 텍스트를 만들어 같은 변환식으로 해석"""
    def __init__(self, dt=600, seed=1):
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)

    def _true_temp(self):
        hour = (self.t / 3600) % 24
        temp = 20 + 7 * np.sin(np.pi * (hour - 9) / 12)   # 15시 최고, 3시 최저
        if 13 <= hour < 15:
            temp += 3                                      # 시나리오: 직사광선으로 일시 과열
        return temp

    def read(self):
        temp = self._true_temp() + self.rng.normal(0, 0.2)
        temp = round(temp / 0.0625) * 0.0625                # 12비트 분해능
        crc = "YES" if self.rng.random() > 0.01 else "NO"  # 1% 통신 오류
        text = (f"72 01 4b 46 7f ff 0e 10 57 : crc=57 {crc}\n"
                f"72 01 4b 46 7f ff 0e 10 57 t={int(temp * 1000)}")
        self.t += self.dt
        return {"temp_c": parse_w1_slave(text)}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualDS18B20(dt=600)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
10분 간격으로 24시간(144회) 측정합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 144               # 측정 횟수
INTERVAL = 600               # 측정 간격(초)

sensor = RealDS18B20() if USE_REAL_SENSOR else VirtualDS18B20(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
1. CRC 오류로 버려진 값(`None`)은 앞뒤 값으로 **보간**합니다.
2. 28°C 이상인 구간을 **과열**로 판정합니다.

In [ ]:
df["hour"] = df["time_s"] / 3600
missing = df["temp_c"].isna().sum()
print(f"통신 오류로 버린 값: {missing}개 → 앞뒤 값으로 보간")
df["temp_c"] = df["temp_c"].interpolate().bfill()

HOT = 28.0
hot = df[df["temp_c"] >= HOT]
print(f"최고 {df['temp_c'].max():.2f}°C ({df.loc[df['temp_c'].idxmax(), 'hour']:.1f}시), "
      f"최저 {df['temp_c'].min():.2f}°C ({df.loc[df['temp_c'].idxmin(), 'hour']:.1f}시)")
if len(hot):
    print(f"과열 경보: {hot['hour'].min():.1f}시 ~ {hot['hour'].max():.1f}시, 총 {len(hot) * INTERVAL / 60:.0f}분")
else:
    print("과열 구간 없음")

## 8. 시각화

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df["hour"], df["temp_c"], "k-", lw=1.5, label="측정 온도")
plt.axhline(HOT, color="gray", ls="--", label=f"과열 기준 {HOT}°C")
plt.fill_between(df["hour"], HOT, df["temp_c"], where=df["temp_c"] >= HOT, color="lightgray")
plt.title("DS18B20 온도 측정 (24시간)")
plt.xlabel("시각 (시)"); plt.ylabel("온도 (°C)")
plt.xticks(range(0, 25, 3)); plt.grid(alpha=0.3); plt.legend()
plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "온도센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "DS18B20",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/temperature", payload)
# client.disconnect()

## 직접 해 보기
- 가상 센서의 통신 오류 확률을 10%로 올리고 보간 결과를 확인해 보세요.
- 측정 간격을 1분(`INTERVAL = 60`, `N_SAMPLES = 1440`)으로 바꿔 보세요.
- 섭씨 온도를 화씨로 바꾸는 열(`temp_f`)을 추가해 보세요. (°F = °C × 9/5 + 32)